# # 🚀 REINFORCE Training — Eco-Scale RL Summative
# Manual REINFORCE (Monte Carlo Policy Gradient) with 10 hyperparameter experiments.
# REINFORCE is NOT in stable-baselines3 — we implement it from scratch using PyTorch.


## Install dependencies


In [ ]:
# !pip install -q gymnasium==0.29.1 numpy matplotlib pandas torch


## Environment definition (embedded)


In [ ]:
import gymnasium as gym
import numpy as np
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

class KubernetesEnv(gym.Env):
    """Custom Gymnasium environment simulating a Kubernetes cluster for pod autoscaling."""
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, trace_type="cyclical", render_mode=None, max_steps=288):
        super().__init__()
        self.trace_type = trace_type
        self.render_mode = render_mode
        self.max_steps = max_steps

        self.observation_space = gym.spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.action_space = gym.spaces.Discrete(3)

        self.alpha = 0.5
        self.beta = 0.3
        self.gamma_r = 0.05

        self.pod_count = 3
        self.current_step = 0
        self.time_of_day = 0
        self.cpu_util = 0.1
        self.request_queue = 50
        self.latency = 0.0
        self.breach_count = 0
        self.episode_reward = 0.0
        self.latency_history = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pod_count = 3
        self.current_step = 0
        self.time_of_day = 0
        self.cpu_util = 0.1
        self.request_queue = 50
        self.latency = 0.0
        self.breach_count = 0
        self.episode_reward = 0.0
        self.latency_history = []
        return self._get_obs(), {}

    def step(self, action):
        if action == 0:
            self.pod_count = max(1, self.pod_count - 1)
        elif action == 2:
            self.pod_count = min(20, self.pod_count + 1)

        self.current_step += 1
        self.time_of_day = (self.time_of_day + 1) % 24

        if self.trace_type == "burst":
            self.cpu_util = self._get_burst_traffic(self.current_step)
        else:
            self.cpu_util = self._get_traffic(self.current_step)

        self.request_queue = int(self.cpu_util * 500)
        reward = self._calculate_reward(action)
        self.episode_reward += reward

        terminated = self._check_termination()
        if terminated:
            reward -= 10.0
        truncated = self.current_step >= self.max_steps

        obs = self._get_obs()
        info = {
            "latency": self.latency, "pods": self.pod_count,
            "step": self.current_step, "cpu_util": self.cpu_util,
            "request_queue": self.request_queue, "time_of_day": self.time_of_day,
            "episode_reward": self.episode_reward,
            "wasted_pods": self._get_wasted_pods(),
        }
        return obs, reward, terminated, truncated, info

    def _get_obs(self):
        return np.array([
            float(self.cpu_util), float(self.pod_count) / 20.0,
            float(self.request_queue) / 1000.0, float(self.time_of_day) / 23.0,
        ], dtype=np.float32)

    def _calculate_reward(self, action):
        self.latency = min(self.request_queue / (self.pod_count * 50.0), 1.0)
        self.latency_history.append(self.latency)
        wasted_pods = self._get_wasted_pods()
        scaling_cost = 1.0 if action != 1 else 0.0
        prev_latency = self.latency_history[-2] if len(self.latency_history) >= 2 else self.latency
        improvement = max(0.0, prev_latency - self.latency)
        return -(self.alpha * self.latency) - (self.beta * wasted_pods) - (self.gamma_r * scaling_cost) + (0.3 * improvement)

    def _get_wasted_pods(self):
        required_pods = max(1, int(np.ceil(self.request_queue / 50.0)))
        return max(0, self.pod_count - required_pods) / 20.0

    def _check_termination(self):
        if self.latency >= 1.0:
            self.breach_count += 1
        else:
            self.breach_count = 0
        return self.breach_count >= 3

    def _get_traffic(self, step):
        hour = step % 24
        if 6 <= hour <= 22:
            base = 0.3 + 0.5 * np.sin((hour - 6) * np.pi / 12)
        else:
            base = 0.1
        noise = self.np_random.normal(0, 0.05)
        return float(np.clip(base + noise, 0.05, 1.0))

    def _get_burst_traffic(self, step):
        base = self._get_traffic(step)
        if self.np_random.random() < 0.1:
            spike = self.np_random.uniform(0.4, 0.7)
            return float(np.clip(base + spike, 0.0, 1.0))
        return base

env = KubernetesEnv()
obs, _ = env.reset()
print(f"✅ Environment ready. Obs: {obs}")


## REINFORCE Implementation


In [ ]:
class PolicyNetwork(nn.Module):
    """Simple MLP policy for discrete action space."""
    def __init__(self, obs_dim=4, act_dim=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim),
        )

    def forward(self, x):
        return Categorical(logits=self.net(x))


class ValueNetwork(nn.Module):
    """Baseline network for variance reduction."""
    def __init__(self, obs_dim=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class REINFORCE:
    """Vanilla REINFORCE with optional learned baseline."""

    def __init__(self, obs_dim=4, act_dim=3, hidden=64,
                 learning_rate=1e-3, gamma=0.99, entropy_coef=0.01,
                 use_baseline=True):
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.use_baseline = use_baseline

        self.policy = PolicyNetwork(obs_dim, act_dim, hidden)
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)

        if use_baseline:
            self.value = ValueNetwork(obs_dim, hidden)
            self.value_optimizer = optim.Adam(self.value.parameters(), lr=learning_rate)

    def select_action(self, obs):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        dist = self.policy(obs_t)
        action = dist.sample()
        return action.item(), dist.log_prob(action), dist.entropy()

    def predict(self, obs, deterministic=False):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        dist = self.policy(obs_t)
        action = dist.probs.argmax(dim=-1) if deterministic else dist.sample()
        return action.item(), None

    def compute_returns(self, rewards):
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        return returns

    def update(self, episode_data):
        obs_list, log_probs, entropies, rewards = episode_data
        returns = self.compute_returns(rewards)

        if self.use_baseline:
            obs_t = torch.FloatTensor(np.array(obs_list))
            values = self.value(obs_t)
            advantages = returns - values.detach()
        else:
            advantages = returns

        policy_loss = sum(-lp * adv for lp, adv in zip(log_probs, advantages)) / len(log_probs)
        entropy_bonus = -self.entropy_coef * sum(entropies) / len(entropies)
        total_loss = policy_loss + entropy_bonus

        self.policy_optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=0.5)
        self.policy_optimizer.step()

        if self.use_baseline:
            obs_t = torch.FloatTensor(np.array(obs_list))
            values = self.value(obs_t)
            value_loss = nn.functional.mse_loss(values, returns)
            self.value_optimizer.zero_grad()
            value_loss.backward()
            self.value_optimizer.step()

        return policy_loss.item(), (sum(entropies) / len(entropies)).item()

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save({
            "policy_state_dict": self.policy.state_dict(),
            "value_state_dict": self.value.state_dict() if self.use_baseline else None,
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, weights_only=False)
        self.policy.load_state_dict(checkpoint["policy_state_dict"])
        if self.use_baseline and checkpoint["value_state_dict"]:
            self.value.load_state_dict(checkpoint["value_state_dict"])


print("✅ REINFORCE agent defined")


## Hyperparameter configs


In [ ]:
CONFIGS = [
    dict(learning_rate=1e-3, gamma=0.99, entropy_coef=0.01, use_baseline=True,
         hidden=64, notes="Baseline"),
    dict(learning_rate=3e-3, gamma=0.99, entropy_coef=0.01, use_baseline=True,
         hidden=64, notes="Higher LR"),
    dict(learning_rate=5e-4, gamma=0.99, entropy_coef=0.01, use_baseline=True,
         hidden=64, notes="Lower LR"),
    dict(learning_rate=1e-3, gamma=0.95, entropy_coef=0.01, use_baseline=True,
         hidden=64, notes="Lower gamma"),
    dict(learning_rate=1e-3, gamma=0.99, entropy_coef=0.05, use_baseline=True,
         hidden=64, notes="More entropy"),
    dict(learning_rate=1e-3, gamma=0.99, entropy_coef=0.001, use_baseline=True,
         hidden=64, notes="Less entropy"),
    dict(learning_rate=1e-3, gamma=0.99, entropy_coef=0.01, use_baseline=False,
         hidden=64, notes="No baseline"),
    dict(learning_rate=1e-3, gamma=0.99, entropy_coef=0.01, use_baseline=True,
         hidden=128, notes="Larger network"),
    dict(learning_rate=1e-3, gamma=0.999, entropy_coef=0.01, use_baseline=True,
         hidden=64, notes="Higher gamma"),
    dict(learning_rate=2e-3, gamma=0.98, entropy_coef=0.02, use_baseline=True,
         hidden=128, notes="Combined changes"),
]

TOTAL_EPISODES = 2000
EVAL_EPISODES = 10

os.makedirs("models/pg", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print(f"📋 {len(CONFIGS)} REINFORCE configurations ready")


## Train all 10 REINFORCE runs


In [ ]:
all_histories = {}
results = []

for i, cfg in enumerate(CONFIGS, start=1):
    notes = cfg["notes"]
    print(f"\n{'='*60}")
    print(f"  REINFORCE Run {i}/10 — {notes}")
    print(f"{'='*60}")

    env = KubernetesEnv(trace_type="cyclical")
    eval_env = KubernetesEnv(trace_type="cyclical")

    agent = REINFORCE(
        obs_dim=4, act_dim=3, hidden=cfg["hidden"],
        learning_rate=cfg["learning_rate"], gamma=cfg["gamma"],
        entropy_coef=cfg["entropy_coef"], use_baseline=cfg["use_baseline"],
    )

    reward_history = []
    best_mean = -float("inf")

    for ep in range(1, TOTAL_EPISODES + 1):
        obs, _ = env.reset()
        obs_list, log_probs, entropies, rewards_ep = [], [], [], []
        done = False

        while not done:
            action, lp, ent = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            obs_list.append(obs)
            log_probs.append(lp)
            entropies.append(ent)
            rewards_ep.append(reward)
            obs = next_obs
            done = terminated or truncated

        episode_reward = sum(rewards_ep)
        reward_history.append(episode_reward)
        loss, ent_val = agent.update((obs_list, log_probs, entropies, rewards_ep))

        if ep % 50 == 0:
            avg50 = np.mean(reward_history[-50:])
            print(f"  Ep {ep:4d} | Reward: {episode_reward:8.2f} | Avg50: {avg50:8.2f}")

        if ep >= 50:
            recent = np.mean(reward_history[-50:])
            if recent > best_mean:
                best_mean = recent
                agent.save(f"models/pg/eco_scale_reinforce_run_{i}_best.pt")

    agent.save(f"models/pg/eco_scale_reinforce_run_{i}.pt")
    all_histories[i] = reward_history

    # Evaluate
    eval_rewards = []
    for _ in range(EVAL_EPISODES):
        obs, _ = eval_env.reset()
        total = 0.0
        done = False
        while not done:
            action, _ = agent.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            total += reward
            done = terminated or truncated
        eval_rewards.append(total)

    mean_r, std_r = float(np.mean(eval_rewards)), float(np.std(eval_rewards))
    print(f"  → Mean Reward: {mean_r:.2f} ± {std_r:.2f}")

    results.append({
        "Run": i, "learning_rate": cfg["learning_rate"],
        "gamma": cfg["gamma"], "entropy_coef": cfg["entropy_coef"],
        "use_baseline": cfg["use_baseline"], "hidden": cfg["hidden"],
        "Mean Reward": round(mean_r, 2), "Std Reward": round(std_r, 2),
        "Notes": notes,
    })

df = pd.DataFrame(results)
df.to_csv("outputs/reinforce_results.csv", index=False)
print(f"\n✅ REINFORCE training complete! Results:")
print(df.to_string(index=False))


## Plot REINFORCE results


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("REINFORCE Hyperparameter Tuning Results", fontsize=14, fontweight="bold")

colors = ["#4CAF50" if r == df["Mean Reward"].max() else "#FF9800" for r in df["Mean Reward"]]
ax1.bar(df["Run"], df["Mean Reward"], yerr=df["Std Reward"], color=colors, alpha=0.8, edgecolor="white")
ax1.set_xlabel("Run")
ax1.set_ylabel("Mean Episode Reward")
ax1.set_title("Mean Reward by Run")
ax1.set_xticks(df["Run"])
ax1.grid(axis="y", alpha=0.3)

ax2.barh([f"R{r} ({n})" for r, n in zip(df["Run"], df["Notes"])],
         df["Mean Reward"], color=colors, alpha=0.8, edgecolor="white")
ax2.set_xlabel("Mean Episode Reward")
ax2.set_title("Hyperparameter Effect")
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/reinforce_hyperparameter_tuning.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved")


## Plot REINFORCE training curves (episode rewards over training)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("REINFORCE Training Curves (All Runs)", fontweight="bold")

for run_id, history in all_histories.items():
    # Smooth with rolling window
    window = 20
    smoothed = pd.Series(history).rolling(window=window, min_periods=1).mean()
    ax.plot(smoothed, label=f"Run {run_id}", alpha=0.7)

ax.set_xlabel("Episode")
ax.set_ylabel("Episode Reward (smoothed)")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/reinforce_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training curves saved")


## Download models and results


In [ ]:
# Works on BOTH Colab and Kaggle
!zip -r reinforce_output.zip models/pg/eco_scale_reinforce_* outputs/reinforce_results.csv outputs/reinforce_*.png

try:
    from google.colab import files
    files.download('reinforce_output.zip')
    print('✅ Download started (Colab)')
except ImportError:
    print('✅ reinforce_output.zip is ready!')
    print('   Kaggle: Click Save Version → Output tab to download')
